# Week 3 · Research collaboration and evidence audit
Advanced Machine Learning · September 15, 2026

This supplementary lab was added to the retrospective study guide. The data are fictional; no actual research results or platform records are used. No account, token or upload is required. Save a copy and use a CPU runtime. Learn to inspect provenance, paired comparisons, qualified claims and a small audit interface. Unfinished exercises do not block the demonstration.

The shared module is fixed by revision and SHA-256. It has no platform integration. Passing its checks does not certify a paper. See the [lecture](https://github.com/lunalab-ai/advML/blob/2026-fall-w3-research/course/notion/w3/w3-research-methodology.md).


In [ ]:
from pathlib import Path
import hashlib, importlib.util, sys, tempfile, urllib.request, subprocess, os, json
os.environ["GRADIO_ANALYTICS_ENABLED"]="False"
if importlib.util.find_spec("gradio") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet","gradio==6.27.0"])
CODE_REF="2026-fall-w3-research"
EXPECTED_SHA256="2b687bbec7a0ffd5a9206c9ce9a1b264a5e82b599cf54ff4df097c4351b6b2e2"
CODE_URL=f"https://raw.githubusercontent.com/lunalab-ai/advML/{CODE_REF}/src/advml_research.py"
local=Path("src/advml_research.py")
if local.is_file() and hashlib.sha256(local.read_bytes()).hexdigest()==EXPECTED_SHA256:
    module_path=local.resolve()
else:
    module_path=Path(tempfile.gettempdir())/f"advml_research_{EXPECTED_SHA256[:16]}.py"
    if not module_path.is_file() or hashlib.sha256(module_path.read_bytes()).hexdigest()!=EXPECTED_SHA256:
        for attempt in range(3):
            try:
                with urllib.request.urlopen(CODE_URL,timeout=30) as response: payload=response.read()
                if hashlib.sha256(payload).hexdigest()!=EXPECTED_SHA256:
                    raise RuntimeError("Course code checksum mismatch; stop and report.")
                module_path.write_bytes(payload)
                break
            except Exception as exc:
                if attempt==2: raise RuntimeError("Fixed course code unavailable. Check connection and rerun, or use a complete public checkout offline.") from exc
spec=importlib.util.spec_from_file_location("advml_research",module_path)
research=importlib.util.module_from_spec(spec);sys.modules["advml_research"]=research;spec.loader.exec_module(research)
import gradio as gr
from IPython.display import display
IN_COLAB="google.colab" in sys.modules
print({"Python":sys.version.split()[0],"Gradio":gr.__version__,"code":CODE_REF,"sha256":EXPECTED_SHA256})


## Shared API
`synthetic_runs()` returns fresh fictional paired rows: seed, a and b (errors; lower is better).
`paired_summary(rows)` requires at least two unique integer seeds and finite nonnegative errors. It returns differences a-b, mean_difference, sample_sd, wins_b and n. Positive differences favor B. It does not calculate significance or generalization.
`example_record()` returns a fresh fictional metadata record and runs.
`audit_record(record)` returns metadata_complete, missing_fields, comparison and limitations; invalid rows raise ValueError. Completeness checks nonempty strings, not their truth.
`audit_view(text)` accepts JSON text and returns findings or an error object.
`build_audit_app()` constructs a Gradio Blocks object without launching it.
All functions are read-only with respect to inputs, filesystem and network. Only setup downloads the fixed source. Read function docstrings for exact fields and errors.


## Predict, then inspect
Will a lower mean error establish universal superiority? Which fields would let someone rerun the study?


In [ ]:
record=research.example_record()
rows=record["runs"]
display(record)


## E1 · Direction and pairing
Complete the mean of a-b across matched rows. Explain why independently sorting each method by error is wrong. Leave None to continue without a solution.


In [ ]:
e1_mean=None  # sum of paired a-b divided by number of rows


## E2 · Missing evidence
The following copy lacks two fields. Identify them and write a retrieval request. Do not invent identifiers.


In [ ]:
incomplete=research.example_record()
incomplete.pop("dataset_version")
incomplete.pop("code_revision")
e2_missing=None
e2_request=None


## E3 · Repair a claim
Replace “B is universally better than A” with a supported statement. State one experiment that could challenge it.


In [ ]:
e3_claim=None
e3_followup=None


## E4 · Wire the web app
Construct a Textbox, Button and JSON output in gr.Blocks; connect research.audit_view. Test missing fields and malformed JSON. The demonstration below works independently of this exercise.


In [ ]:
e4_app=None  # create the components and connect the callback


## Completed demonstration
Inspect both settings before launching. This app does not access LUNA or save records. Colab may create a temporary public endpoint; do not enter private data. Set LAUNCH_APP=True to open it in Colab; local users may call demo.launch() themselves.


In [ ]:
display(research.audit_record(record))
display(research.audit_record(incomplete))
demo=research.build_audit_app()
LAUNCH_APP=False
if IN_COLAB and LAUNCH_APP:
    demo.launch(share=True,prevent_thread_lock=True)
else:
    print("App constructed. Numerical and callback checks run without a public endpoint.")


In [ ]:
import math, copy
before=copy.deepcopy(record)
checked=research.audit_record(record)
assert checked["metadata_complete"] and record==before
assert checked["comparison"]["n"]==len(rows)
assert 0<=checked["comparison"]["wins_b"]<=len(rows)
assert math.isfinite(checked["comparison"]["mean_difference"])
assert checked["comparison"]["sample_sd"]>=0
assert research.paired_summary(list(reversed(rows)))["mean_difference"]==checked["comparison"]["mean_difference"]
assert set(research.audit_view(json.dumps(incomplete))["missing_fields"])=={"dataset_version","code_revision"}
for bad in ["broken", "[]", '{"runs":[]}']:
    assert "error" in research.audit_view(bad)
for badrows in [[rows[0],rows[0]], [{"seed":0,"a":float("nan"),"b":.1},rows[1]]]:
    try: research.paired_summary(badrows)
    except ValueError: pass
    else: raise AssertionError("Invalid comparison accepted")
print("W3 paired arithmetic, provenance, error handling and callback checks passed.")


## Checkpoint questions
1. Why does closing a research issue not establish a scientific claim?

2. What should you check when a document is missing from a project view?

3. Why compare methods using matched run identifiers?

4. What does a metadata audit fail to prove?

5. When is an AI assistant right to withhold a research-readiness judgment?

6. How will this workflow support the Week 4 variational-inference experiment?
